# Evaluate Retrieval Runs with Stable Ground Truth Qrels

This notebook is responsible only for evaluation. It assumes notebook 2 has already exposed the stable qrels file:

```text
groundtruth_outputs/qrels/qrels_500q_top50.txt
```

It does not build candidate pools, call an LLM judge, or validate human agreement.

## Notebook Setup

Expected TREC run format:

```text
query_id Q0 doc_id rank score method_name
```

All `.trec` files placed in `groundtruth_outputs/runs` can be evaluated against the stable qrels file.

In [ ]:
from __future__ import annotations

import math
from collections import defaultdict
from pathlib import Path
from typing import Optional

import pandas as pd


def find_finalproject_root(start_path: Optional[Path] = None) -> Path:
    """Find the Finalproject directory by walking upward from the current path."""
    current_path = (start_path or Path.cwd()).resolve()
    for candidate_path in [current_path, *current_path.parents]:
        if candidate_path.name == "Finalproject":
            return candidate_path
        nested_finalproject = candidate_path / "Finalproject"
        if nested_finalproject.exists() and nested_finalproject.is_dir():
            return nested_finalproject.resolve()
    raise FileNotFoundError("Could not find the Finalproject directory.")


FINALPROJECT_ROOT = find_finalproject_root()
NOTEBOOK_OUTPUT_DIR = FINALPROJECT_ROOT / "notebooks" / "rec_and_eval" / "groundtruth_outputs"
QRELS_PATH = NOTEBOOK_OUTPUT_DIR / "qrels" / "qrels_500q_top50.txt"
RUNS_DIR = NOTEBOOK_OUTPUT_DIR / "runs"
METRICS_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / "metrics"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_CUTOFFS = [10, 20, 50]

print("Qrels path:", QRELS_PATH)
print("Runs directory:", RUNS_DIR)
print("Metrics output directory:", METRICS_OUTPUT_DIR)

## Load Qrels and Runs

This section provides loaders for stable qrels and TREC run files. The qrels labels are graded from 0 to 3.

In [ ]:
def load_trec_qrels(qrels_path: Path) -> dict[int, dict[int, int]]:
    """Load TREC qrels into query_id -> doc_id -> relevance."""
    qrels = defaultdict(dict)
    with qrels_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if not line.strip():
                continue
            query_id_text, _, doc_id_text, relevance_text = line.strip().split()[:4]
            qrels[int(query_id_text)][int(doc_id_text)] = int(relevance_text)
    return qrels


def load_trec_run(run_path: Path) -> dict[int, list[tuple[int, float]]]:
    """Load a TREC run file into query_id -> ranked list of (doc_id, score)."""
    run = defaultdict(list)
    with run_path.open("r", encoding="utf-8") as input_file:
        for line in input_file:
            if not line.strip():
                continue
            parts = line.strip().split()
            query_id = int(parts[0])
            doc_id = int(parts[2])
            rank = int(parts[3])
            score = float(parts[4])
            run[query_id].append((doc_id, score, rank))

    sorted_run = {}
    for query_id, documents in run.items():
        sorted_documents = sorted(documents, key=lambda item: (item[2], -item[1]))
        sorted_run[query_id] = [(doc_id, score) for doc_id, score, _ in sorted_documents]
    return sorted_run


qrels = load_trec_qrels(QRELS_PATH) if QRELS_PATH.exists() else {}
print("Loaded qrels for queries:", len(qrels))

## Metric Functions

Because labels are graded, `nDCG@K` should be treated as the primary metric. Binary metrics such as precision, recall, and reciprocal rank treat any label greater than zero as relevant.

In [ ]:
def dcg_at_k(relevance_values: list[int], k: int) -> float:
    """Compute discounted cumulative gain at K using graded relevance values."""
    return sum(relevance / math.log2(rank + 1) for rank, relevance in enumerate(relevance_values[:k], start=1))


def ndcg_at_k_for_query(ranked_doc_ids: list[int], relevance_by_doc_id: dict[int, int], k: int) -> float:
    """Compute nDCG@K for one query."""
    predicted_relevance = [relevance_by_doc_id.get(doc_id, 0) for doc_id in ranked_doc_ids[:k]]
    ideal_relevance = sorted(relevance_by_doc_id.values(), reverse=True)[:k]
    ideal_dcg = dcg_at_k(ideal_relevance, k)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(predicted_relevance, k) / ideal_dcg


def precision_at_k_for_query(ranked_doc_ids: list[int], relevance_by_doc_id: dict[int, int], k: int) -> float:
    """Compute binary Precision@K where any relevance > 0 is relevant."""
    if k == 0:
        return 0.0
    hit_count = sum(1 for doc_id in ranked_doc_ids[:k] if relevance_by_doc_id.get(doc_id, 0) > 0)
    return hit_count / k


def recall_at_k_for_query(ranked_doc_ids: list[int], relevance_by_doc_id: dict[int, int], k: int) -> float:
    """Compute binary Recall@K where any relevance > 0 is relevant."""
    relevant_doc_ids = {doc_id for doc_id, relevance in relevance_by_doc_id.items() if relevance > 0}
    if not relevant_doc_ids:
        return 0.0
    retrieved_relevant_count = sum(1 for doc_id in ranked_doc_ids[:k] if doc_id in relevant_doc_ids)
    return retrieved_relevant_count / len(relevant_doc_ids)


def reciprocal_rank_at_k_for_query(ranked_doc_ids: list[int], relevance_by_doc_id: dict[int, int], k: int) -> float:
    """Compute reciprocal rank at K using binary relevance."""
    for rank, doc_id in enumerate(ranked_doc_ids[:k], start=1):
        if relevance_by_doc_id.get(doc_id, 0) > 0:
            return 1.0 / rank
    return 0.0

## Evaluate Runs

The main function returns aggregate metrics for one run. The final cell can evaluate every `.trec` file in the runs directory and save a CSV summary.

In [ ]:
def evaluate_run_with_qrels(
    qrels: dict[int, dict[int, int]],
    run: dict[int, list[tuple[int, float]]],
    cutoffs: list[int],
) -> pd.DataFrame:
    """Evaluate one run with common TREC-style metrics."""
    metric_rows = []
    for query_id, relevance_by_doc_id in qrels.items():
        ranked_doc_ids = [doc_id for doc_id, _ in run.get(query_id, [])]
        for cutoff in cutoffs:
            metric_rows.append(
                {
                    "query_id": query_id,
                    "cutoff": cutoff,
                    "ndcg": ndcg_at_k_for_query(ranked_doc_ids, relevance_by_doc_id, cutoff),
                    "precision": precision_at_k_for_query(ranked_doc_ids, relevance_by_doc_id, cutoff),
                    "recall": recall_at_k_for_query(ranked_doc_ids, relevance_by_doc_id, cutoff),
                    "reciprocal_rank": reciprocal_rank_at_k_for_query(ranked_doc_ids, relevance_by_doc_id, cutoff),
                }
            )
    per_query_metrics = pd.DataFrame(metric_rows)
    return (
        per_query_metrics.groupby("cutoff")[["ndcg", "precision", "recall", "reciprocal_rank"]]
        .mean()
        .reset_index()
    )


def evaluate_all_trec_runs(
    qrels: dict[int, dict[int, int]],
    runs_directory: Path,
    cutoffs: list[int],
) -> pd.DataFrame:
    """Evaluate every .trec run file in a directory."""
    all_metric_frames = []
    for run_path in sorted(runs_directory.glob("*.trec")):
        method_name = run_path.stem
        run = load_trec_run(run_path)
        metrics = evaluate_run_with_qrels(qrels=qrels, run=run, cutoffs=cutoffs)
        metrics.insert(0, "method_name", method_name)
        all_metric_frames.append(metrics)

    if not all_metric_frames:
        return pd.DataFrame(columns=["method_name", "cutoff", "ndcg", "precision", "recall", "reciprocal_rank"])

    return pd.concat(all_metric_frames, ignore_index=True)


# Run after placing .trec files in RUNS_DIR.
# all_metrics = evaluate_all_trec_runs(qrels=qrels, runs_directory=RUNS_DIR, cutoffs=DEFAULT_CUTOFFS)
# metrics_output_path = METRICS_OUTPUT_DIR / "retrieval_metrics.csv"
# all_metrics.to_csv(metrics_output_path, index=False, encoding="utf-8-sig")
# display(all_metrics)